In [ ]:
#Convert the audio file to mono
import os
import torchaudio
import torch
from tqdm import tqdm
from pathlib import Path

def convert_to_mono(filepath, output_path):
    audio, sample_rate = torchaudio.load(filepath)
    if audio.shape[0] == 2:
        mono = torch.mean(audio, dim=0, keepdim=True)
        torchaudio.save(output_path, mono, sample_rate)
    else:
        print(f"The audio at {filepath} is not stereo.")

def convert_folder_to_mono(input_dir, output_dir):
    wav_files = [f for f in os.listdir(input_dir) if f.endswith('.wav')]
    for wav_file in tqdm(wav_files, desc="Converting files"):
        input_path = os.path.join(input_dir, wav_file)
        output_path = os.path.join(output_dir, wav_file)
        convert_to_mono(input_path, output_path)



input_dir = str(Path.cwd().parents[2].joinpath('Audio_data', 'pretrain_Audio'))
output_dir = str(Path.cwd().parents[2].joinpath('Audio_data', 'pretrain_Audio_mono'))
convert_folder_to_mono(input_dir, output_dir)

In [4]:
import csv
from pathlib import Path

input_csv_path = str(Path.cwd().parents[2].joinpath('Audio_data', 'setting', 'species.csv'))

output_csv_path = str(Path.cwd().parents[2].joinpath('Audio_data', 'setting', 'pretrain_labels_indices.csv'))

add_dummy_label = True

with open(input_csv_path, "r") as input_file:
    reader = csv.DictReader(input_file)
    rows = [row for row in reader if row["target"] == "TRUE"]

with open(output_csv_path, "w") as output_file:
    fieldnames = ["index", "mid", "display_name"]
    writer = csv.DictWriter(output_file, fieldnames=fieldnames)
    writer.writeheader()

    if add_dummy_label:
        writer.writerow({"index": 0, "mid": "/m/dummy", "display_name": "dummy"})
        start_index = 1
    else:
        start_index = 0

    for index, row in enumerate(rows, start=start_index):
        writer.writerow({
            "index": index,
            "mid": f"/m/bs{str(index).zfill(2)}",
            "display_name": row["code"]
        })


In [ ]:
#Generate a JSON file for the pretrain training dataset
import os
import json
import random
import librosa
from pathlib import Path
from collections import defaultdict
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
import re

wav_directory = str(Path.cwd().parents[2].joinpath('Audio_data', 'finetune_Audio_mono'))
train_txt_path = str(Path.cwd().parents[2].joinpath('Audio_data', 'pretrain_train_list.txt'))
val_txt_path = str(Path.cwd().parents[2].joinpath('Audio_data', 'pretrain_val_list.txt'))
pretrain_train_path = str(Path.cwd().parents[2].joinpath('temporary_file', 'pretrain_train.json'))
custom_output_paths = [
    str(Path.cwd().parents[2].joinpath('Audio_data', 'pretrain_train.json')),
    str(Path.cwd().parents[2].joinpath('Audio_data', 'pretrain_val.json'))
]

sliding_second = 1
segment_second = 1
num_processes = cpu_count()

def process_audio_files():
    wav_files = os.listdir(wav_directory)
    regex = re.compile(r"([A-Z0-9]+)_(\d{8})_\d{6}\.wav")
    counts = defaultdict(lambda: defaultdict(int))
    filenames = defaultdict(lambda: defaultdict(list))
    selected_counts_1 = defaultdict(lambda: defaultdict(int))
    selected_counts_2 = defaultdict(lambda: defaultdict(int))
    copied_files_1, copied_files_2, remaining_files = [], [], []
    total = 0

    for file in wav_files:
        match = regex.match(file)
        if match:
            station, date = match.group(1), match.group(2)
            counts[station][date] += 1
            filenames[station][date].append(file)
            total += 1
        else:
            print(f"Skipped file: {file}")

    pick_probability_1, pick_probability_2 = 0.2, 0.0
    all_picked_files = set()

    for station, dates_files in filenames.items():
        for date, files in dates_files.items():
            total_files = len(files)
            num_to_sample_1 = int(total_files * pick_probability_1)
            num_to_sample_2 = int(total_files * pick_probability_2)
            sampled_files = random.sample(files, min(total_files, num_to_sample_1 + num_to_sample_2))
            sampled_files_1 = sampled_files[:num_to_sample_1]
            sampled_files_2 = sampled_files[num_to_sample_1:num_to_sample_1 + num_to_sample_2]
            copied_files_1.extend(sampled_files_1)
            copied_files_2.extend(sampled_files_2)
            all_picked_files.update(sampled_files_1)
            all_picked_files.update(sampled_files_2)

    all_files = [os.path.splitext(file)[0] for file in wav_files]
    remaining_files = list(set(all_files) - all_picked_files)  # Exclude validation and test files from training set

    with open(val_txt_path, "w") as file:
        file.write("\n".join([os.path.splitext(f)[0] for f in copied_files_1]))
    with open(train_txt_path, "w") as file:
        file.write("\n".join(remaining_files))

def process_files(file_chunk):
    results = []
    for file_name in tqdm(file_chunk):
        if file_name.endswith(".wav"):
            audio_file_path = os.path.join(wav_directory, file_name)
            audio_data, sr = librosa.load(audio_file_path, sr=None)

            start_time = 0
            end_time = segment_second * sr
            step_size = int(sr * sliding_second)

            while end_time <= len(audio_data):
                results.append({
                    "wav": audio_file_path,
                    "start_time": start_time / sr,
                    "end_time": end_time / sr,
                    "labels": "/m/dummy"
                })
                start_time += step_size
                end_time += step_size
    return results

def generate_segments():
    files_per_process = len(os.listdir(wav_directory)) // num_processes + 1
    file_chunks = [os.listdir(wav_directory)[i:i+files_per_process] for i in range(0, len(os.listdir(wav_directory)), files_per_process)]
    with Pool(processes=num_processes) as pool:
        results = pool.map(process_files, file_chunks)
    all_segments = [result for sublist in results for result in sublist]
    random.shuffle(all_segments)
    with open(pretrain_train_path, "w") as f:
        json.dump({"data": all_segments}, f, indent=2)

def load_json_data(json_file):
    with open(json_file, 'r') as file:
        data = json.load(file)
    return data['data']

def load_txt_data(txt_file):
    with open(txt_file, 'r') as file:
        filenames = [line.strip() for line in file.readlines()]
    return filenames

def match_data(json_data, txt_filenames):
    matched_data = []
    for entry in json_data:
        basename = entry['wav'].split('/')[-1].split('.')[0]
        if basename in txt_filenames:
            matched_data.append(entry)
    return matched_data

def write_json_data(output_file, data):
    with open(output_file, 'w') as file:
        json.dump({"data": data}, file, indent=2)

def finalize_data(custom_output_paths):
    json_data = load_json_data(pretrain_train_path)
    txt_files = [train_txt_path, val_txt_path]
    
    for txt_filename, output_path in zip(txt_files, custom_output_paths):
        txt_data = load_txt_data(txt_filename)
        matched_data = match_data(json_data, txt_data)
        write_json_data(output_path, matched_data)

process_audio_files()
generate_segments()
finalize_data(custom_output_paths)

In [ ]:
## Calculate mean and std
import torch
import numpy as np
from tqdm import tqdm
from src.dataset import AudiosetDataset
from pathlib import Path

audio_conf = {'num_mel_bins': 128,
              'target_length': 128,
              'freqm': 0,
              'timem': 0,
              'mixup': 0,
              'dataset': 'birdsong',
              'mode': 'train',
              'skip_norm': True, 
              'multilabel': True,
              'noise':False,
              'frame_shift_ms': 10,
              'low_freq': 100,
              'high_freq': 11000}

data_path = str(Path.cwd().parents[2].joinpath('Audio_data', 'pretrain_train.json'))
lable_path = str(Path.cwd().parents[2].joinpath('Audio_data', 'pretrain_labels_indices.csv'))

train_loader = torch.utils.data.DataLoader(
    AudiosetDataset(data_path, label_csv=lable_path, audio_conf=audio_conf),
                            batch_size=65536, shuffle=False, num_workers=8, pin_memory=True)


mean=[]
std=[]
for i, (audio_input, labels, _) in tqdm(enumerate(train_loader), total=len(train_loader)):
    cur_mean = torch.mean(audio_input)
    cur_std = torch.std(audio_input)
    mean.append(cur_mean)
    std.append(cur_std)
    # print(cur_mean, cur_std)

print(np.mean(mean), np.mean(std))
